# Module 11: SQL for Data People

**Duration:** 16 hours  
**ML Focus:** Feature Extraction from Relational Databases

SQL is essential for data scientists. Most production data lives in relational databases, and efficient SQL skills let you extract, aggregate, and join data directly — often faster and more efficiently than pandas.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Create in-memory database for this lesson
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
print('SQLite database created in memory.')

## 1. Creating Tables and Inserting Data

We'll create a mini relational schema: customers, orders, and products.

In [ ]:
# Create tables
cur.execute('''
    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        age INTEGER,
        city TEXT,
        signup_date DATE
    )
''')

cur.execute('''
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        order_date DATE,
        amount REAL,
        category TEXT,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
    )
''')

cur.execute('''
    CREATE TABLE products (
        product_id INTEGER PRIMARY KEY,
        product_name TEXT,
        price REAL,
        category TEXT
    )
''')

print('Tables created successfully.')

In [ ]:
# Insert sample data
customers_data = [
    (1, 'Alice Johnson', 32, 'New York', '2023-01-15'),
    (2, 'Bob Smith', 45, 'Los Angeles', '2023-03-22'),
    (3, 'Charlie Brown', 28, 'Chicago', '2023-06-10'),
    (4, 'Diana Prince', 36, 'New York', '2023-02-01'),
    (5, 'Eve Davis', 51, 'Houston', '2023-05-18')
]

orders_data = [
    (1, 1, '2024-01-10', 150.0, 'Electronics'),
    (2, 1, '2024-02-15', 75.0, 'Books'),
    (3, 1, '2024-03-20', 200.0, 'Electronics'),
    (4, 2, '2024-01-25', 50.0, 'Clothing'),
    (5, 2, '2024-04-10', 120.0, 'Home'),
    (6, 3, '2024-02-05', 300.0, 'Electronics'),
    (7, 3, '2024-05-12', 45.0, 'Books'),
    (8, 4, '2024-03-01', 88.0, 'Clothing'),
    (9, 4, '2024-04-18', 210.0, 'Home'),
    (10, 4, '2024-06-01', 65.0, 'Books'),
    (11, 5, '2024-02-20', 500.0, 'Electronics'),
    (12, 5, '2024-05-05', 35.0, 'Food')
]

# Use parameterized queries (safe from SQL injection)
cur.executemany(
    'INSERT INTO customers VALUES (?, ?, ?, ?, ?)',
    customers_data
)
cur.executemany(
    'INSERT INTO orders VALUES (?, ?, ?, ?, ?)',
    orders_data
)
conn.commit()
print(f'Inserted {len(customers_data)} customers and {len(orders_data)} orders.')

## 2. SELECT, WHERE, GROUP BY, ORDER BY, LIMIT

The fundamental SQL operations for data exploration.

In [ ]:
print('=== Basic SELECT and WHERE ===')
query = '''
    SELECT customer_id, name, age, city
    FROM customers
    WHERE age > 30 AND city = 'New York'
    ORDER BY age DESC
'''
df = pd.read_sql(query, conn)
print(df)

print('\n=== GROUP BY and Aggregation ===')
query = '''
    SELECT category,
           COUNT(*) AS order_count,
           ROUND(SUM(amount), 2) AS total_revenue,
           ROUND(AVG(amount), 2) AS avg_amount,
           MAX(amount) AS max_amount
    FROM orders
    GROUP BY category
    HAVING COUNT(*) >= 2
    ORDER BY total_revenue DESC
'''
df_grouped = pd.read_sql(query, conn)
print(df_grouped)

print('\n=== LIMIT and OFFSET ===')
query = 'SELECT * FROM orders ORDER BY amount DESC LIMIT 3 OFFSET 1'
df_limited = pd.read_sql(query, conn)
print(df_limited)

## 3. JOIN Operations

Joins are how you combine related tables in SQL.

In [ ]:
print('=== INNER JOIN ===')
query = '''
    SELECT c.name, c.city, o.order_id, o.amount, o.category, o.order_date
    FROM customers c
    INNER JOIN orders o ON c.customer_id = o.customer_id
    ORDER BY o.amount DESC
'''
df_inner = pd.read_sql(query, conn)
print(df_inner.head())
print(f'\nINNER JOIN returned {len(df_inner)} rows')

print('\n=== LEFT JOIN (customers without orders will appear) ===')
query = '''
    SELECT c.name, COUNT(o.order_id) AS order_count, ROUND(SUM(o.amount), 2) AS total_spent
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id
    ORDER BY total_spent DESC
'''
df_left = pd.read_sql(query, conn)
print(df_left)

print('\nNote: SQLite does not support RIGHT/FULL JOIN natively.')
print('RIGHT JOIN = reverse table order in LEFT JOIN.')
print('FULL JOIN = LEFT JOIN UNION ALL LEFT JOIN (reversed).')

## 4. Subqueries

Subqueries allow nested queries for complex filtering and derived calculations.

In [ ]:
print('=== Scalar Subquery in WHERE ===')
query = '''
    SELECT name, age
    FROM customers
    WHERE age > (SELECT AVG(age) FROM customers)
'''
df_scalar = pd.read_sql(query, conn)
print(df_scalar)

print('\n=== Subquery in FROM (derived table) ===')
query = '''
    SELECT c.name, top_spenders.total_spent
    FROM customers c
    JOIN (
        SELECT customer_id, SUM(amount) AS total_spent
        FROM orders
        GROUP BY customer_id
        HAVING SUM(amount) > 200
    ) AS top_spenders ON c.customer_id = top_spenders.customer_id
'''
df_derived = pd.read_sql(query, conn)
print(df_derived)

print('\n=== EXISTS (check for existence) ===')
query = '''
    SELECT name, city
    FROM customers c
    WHERE EXISTS (
        SELECT 1 FROM orders o
        WHERE o.customer_id = c.customer_id AND o.amount > 250
    )
'''
df_exists = pd.read_sql(query, conn)
print(df_exists)

## 5. Common Table Expressions (CTEs)

CTEs make complex queries readable and reusable.

In [ ]:
print('=== CTE: Customer spending tiers ===')
query = '''
    WITH customer_spending AS (
        SELECT c.customer_id, c.name,
               COUNT(o.order_id) AS order_count,
               ROUND(SUM(o.amount), 2) AS total_spent
        FROM customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
        GROUP BY c.customer_id
    ),
    spending_tiers AS (
        SELECT *,
               CASE
                   WHEN total_spent >= 300 THEN 'VIP'
                   WHEN total_spent >= 100 THEN 'Regular'
                   WHEN total_spent > 0 THEN 'Occasional'
                   ELSE 'Inactive'
               END AS tier
        FROM customer_spending
    )
    SELECT * FROM spending_tiers ORDER BY total_spent DESC
'''
df_cte = pd.read_sql(query, conn)
print(df_cte)

## 6. Window Functions

Window functions perform calculations across rows related to the current row without collapsing groups.

In [ ]:
print('=== ROW_NUMBER, RANK, DENSE_RANK ===')
query = '''
    SELECT name, category, amount,
           ROW_NUMBER() OVER (ORDER BY amount DESC) AS row_num,
           RANK() OVER (ORDER BY amount DESC) AS rank,
           DENSE_RANK() OVER (ORDER BY amount DESC) AS dense_rank
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    LIMIT 8
'''
df_window = pd.read_sql(query, conn)
print(df_window)

print('\n=== PARTITION BY: Rank within each category ===')
query = '''
    SELECT name, category, amount,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY amount DESC) AS cat_rank
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    ORDER BY category, cat_rank
'''
df_partition = pd.read_sql(query, conn)
print(df_partition)

print('\n=== LAG (previous value) ===')
query = '''
    SELECT customer_id, order_date, amount,
           LAG(amount, 1) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_amount,
           amount - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS change_from_prev
    FROM orders
    ORDER BY customer_id, order_date
'''
df_lag = pd.read_sql(query, conn)
print(df_lag)

## 7. Pandas vs SQL Comparison

Let's compare equivalent operations side by side.

In [ ]:
# Load all data into pandas
customers_df = pd.read_sql('SELECT * FROM customers', conn)
orders_df = pd.read_sql('SELECT * FROM orders', conn)

print('=== SQL vs Pandas: Filtering ===')
sql_result = pd.read_sql("SELECT name, age FROM customers WHERE age > 30 ORDER BY age", conn)
pd_result = customers_df[customers_df['age'] > 30][['name', 'age']].sort_values('age')
print('SQL result:')
print(sql_result)
print('\nPandas result:')
print(pd_result)

print('\n=== SQL vs Pandas: GroupBy ===')
sql_group = pd.read_sql('''
    SELECT category, COUNT(*) AS cnt, AVG(amount) AS avg_amount
    FROM orders GROUP BY category ORDER BY cnt DESC
''', conn)
pd_group = orders_df.groupby('category').agg(
    cnt=('amount', 'count'),
    avg_amount=('amount', 'mean')
).sort_values('cnt', ascending=False).reset_index()
print('SQL result:', sql_group.to_dict('records'))
print('Pandas result:', pd_group.to_dict('records'))

print('\n=== SQL vs Pandas: Merge/Join ===')
sql_join = pd.read_sql('''
    SELECT c.name, o.order_id, o.amount
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    ORDER BY c.name
''', conn)
pd_join = pd.merge(customers_df, orders_df, on='customer_id', how='left')
pd_join = pd_join[['name', 'order_id', 'amount']].sort_values('name')
print('Both produce the same result.')

## 8. ML Focus: Feature Extraction from Relational Data

This is the most important skill: creating a flat feature matrix from a relational database for ML modeling.

In [ ]:
print('=== Feature Extraction: Customer Profile + Behavioral Features ===')

feature_query = '''
    WITH customer_base AS (
        SELECT customer_id, name, age, city,
               JULIANDAY('2025-01-01') - JULIANDAY(signup_date) AS days_since_signup
        FROM customers
    ),
    order_features AS (
        SELECT
            customer_id,
            COUNT(*) AS total_orders,
            ROUND(SUM(amount), 2) AS total_spent,
            ROUND(AVG(amount), 2) AS avg_order_value,
            ROUND(AVG(amount), 2) AS avg_order_value,
            MAX(order_date) AS last_order_date,
            COUNT(DISTINCT category) AS distinct_categories
        FROM orders
        GROUP BY customer_id
    ),
    category_features AS (
        SELECT customer_id, category AS favorite_category
        FROM (
            SELECT customer_id, category,
                   ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY COUNT(*) DESC) AS rn
            FROM orders
            GROUP BY customer_id, category
        )
        WHERE rn = 1
    )
    SELECT
        cb.customer_id,
        cb.name,
        cb.age,
        cb.city,
        cb.days_since_signup,
        COALESCE(of.total_orders, 0) AS total_orders,
        COALESCE(of.total_spent, 0) AS total_spent,
        COALESCE(of.avg_order_value, 0) AS avg_order_value,
        COALESCE(of.distinct_categories, 0) AS distinct_categories,
        COALESCE(cf.favorite_category, 'None') AS favorite_category,
        CASE
            WHEN of.last_order_date IS NULL THEN 999
            ELSE JULIANDAY('2025-01-01') - JULIANDAY(of.last_order_date)
        END AS days_since_last_order
    FROM customer_base cb
    LEFT JOIN order_features of ON cb.customer_id = of.customer_id
    LEFT JOIN category_features cf ON cb.customer_id = cf.customer_id
    ORDER BY total_spent DESC
'''

features_df = pd.read_sql(feature_query, conn)
print('Feature matrix shape:', features_df.shape)
print(features_df)

print('\n=== ML-Ready Dataset ===')
print('All features are numeric (or can be one-hot encoded):')
print('Numeric columns:', features_df.select_dtypes(include=['int64', 'float64']).columns.tolist())
print('Categorical columns:', features_df.select_dtypes(include=['object']).columns.tolist())
print('\nModule 11 complete! You can now:')
print('  - Query SQL databases efficiently')
print('  - Write joins, subqueries, and CTEs')
print('  - Use window functions for advanced analytics')
print('  - Extract ML-ready feature matrices from relational data')